In [327]:
import fitz
import torch
import base64
import numpy as np
from dotenv import load_dotenv

from colpali_engine.models import ColQwen2, ColQwen2Processor

import psycopg2
from pgvector.psycopg2 import register_vector

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage



In [273]:
pdf_path = "pdfs\supplier_guide_detailed.pdf"

In [274]:
load_dotenv()

True

In [275]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [276]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [277]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [278]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()


Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  7.55s/it]


In [279]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [109]:
def get_coarse_embedding(embeddings_tensor):
    normalized = embeddings_tensor / embeddings_tensor.norm(dim=-1, keepdim=True)
    return normalized.mean(dim=1).squeeze().to(torch.float32).cpu() 

In [280]:
def embed_query(query, model, processor):
    inputs = processor.process_queries(queries=[query])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        embeddings = model(**inputs)
        embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        coarse_vector = get_coarse_embedding(embeddings).cpu().flatten().tolist()
        embeddings = embeddings.squeeze(0).to(torch.float32).cpu().numpy()
    return coarse_vector, embeddings


In [322]:
query = "According to 'Figure 1: Summary of Government Procurement,' what is the percentage value assigned to 'Services"

In [323]:
coarse_vector, embeddings = embed_query(query, model, processor)

In [306]:
def coarse_search(coarse_vector, top_k = 20):
    try:
        with conn.cursor() as cur:
            cur.execute('''
                SELECT id FROM pages
                ORDER BY coarse_vector <=> %s::vector
                LIMIT %s

            ''', (coarse_vector, top_k))
            ids = [row[0] for row in cur.fetchall()]
            return ids
    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [324]:
ids = coarse_search(coarse_vector)

In [308]:
def precision_search(embeddings, ids, top_k = 3):
    try: 
        final_scores = []
        for p_id in ids:
            with conn.cursor() as cur:
                cur.execute('''
                    SELECT patch_embedding
                    FROM page_patches
                    WHERE page_id = %s
                    ORDER BY patch_index
                ''', (p_id,))
            
                rows = cur.fetchall()
                page_matrix = np.array([r[0] for r in rows], dtype = np.float32)
                sim_matrix = embeddings @ page_matrix.T
                page_score = np.sum(np.max(sim_matrix, axis=1))
                final_scores.append((p_id, page_score))
        return sorted(final_scores, key=lambda x: x[1], reverse=True)[:top_k]

    except Exception as e:
        print(f"Error: {e}")
        conn.rollback()

In [325]:
results = precision_search(embeddings, ids)

In [310]:
def retrieve_content(results):
    output = []
    with conn.cursor() as cur:
        for id, score in results:
            cur.execute('''
                SELECT page_number, markdown
                FROM pages
                WHERE id = %s
            ''', (id,))

            row = cur.fetchone()
            output.append({
                "page_number": row[0],
                "markdown": row[1]
            })
    return output

In [326]:
content = retrieve_content(results)
content

[{'page_number': 3,
  'markdown': "## 1. About this Guide\n\nThis guide is useful for suppliers looking to deliver goods or services to the Singapore Government. It will help you gain a better understanding of Singapore's Government procurement processes, avoid common pitfalls and compete effectively for Government contracts.\n\nThis guide addresses key questions such as:\n\n- What does the Government procure?\n- What are the rules and processes of Government procurement?\n- How to look for business opportunities?\n- How to participate?\n- How to submit a competitive bid?\n\n## 2. Understanding Government Procurement\n\n## Government Procurement as Business Opportunities\n\nThe procurement activities of Ministries, Departments, Organs of State, and Statutory Boards are governed by a set of procurement rules that lays out the minimum standards that they have to comply with. Each agency is responsible for making their own procurement decisions and may individually adopt  more  stringent 

In [289]:
def get_page_base64(pdf_path, page_number):
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)

    zoom = 2 
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)

    img_bytes = pix.tobytes("png")
    return base64.b64encode(img_bytes).decode('utf-8')


In [290]:
def generate_context(content, pdf_path):
    context = []
    for dic in content:
        context.append({
            "type": "text",
            "text": f"--- Page {dic['page_number']} ---\n{dic['markdown']}"
        })

        b64_img = get_page_base64(pdf_path, dic["page_number"])

        context.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{b64_img}"}
        })
    return HumanMessage(content=context)

In [291]:
context = generate_context(content, pdf_path)

In [292]:
def generate_message(query, context):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context which includes the markdown and image of pages of a document.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. DO NOT USE EXTERNAL KNOWLEDGE.
    2. MARKDOWN: Use the markdown text for precise data extractions, quotes and tables.
    3. IMAGES: Use the images to verify layout, check for visual elements, and clarify ambiguous parts of the text.
    4. DISCREPANCY: The image is the GROUND TRUTH. Always verify markdown information with the image. If there is a discrepancy between the markdown and the image, TRUST THE IMAGE.
    5. CITATIONS: You MUST CITE THE SPECIFIC PAGE NUMBER (e.g. [Page 4]) for every fact or claim you make. Each page is labelled at the start (e.g. --- Page 4 ---).
    6. CONTEXT INTEGRATION: Treat the context as a single unified knowledge base, even if they are provided out of chronological order.
    7. RELEVANCE: ONLY USE INFORMATION FROM THE CONTEXT that are relevant to answering the question.  
    8. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    context, 
    HumanMessage(content=f"""
    Question: {query}
    """)

    ]
    return messages

In [293]:
messages = generate_message(query, context)

In [294]:
response = llm.invoke(messages)

In [295]:
print(response.content)

According to the chart on page 3, the category that represents only 10% of the government's annual procurement value is "Goods." Referring to the procurement table on page 6, the specific sourcing method most likely used for this category is **Small Value Purchases (SVP)**, which is applicable for estimated procurement values not exceeding S$6,000. 

- **Sourcing Method for Goods (10% of procurement value)**: 
  - **SVP**: Buy directly from suitable suppliers or off-the-shelf, provided the prices reflect fair market value [Page 6].


TESTING